# Spotify Song Recommender
Reconstructed from a scrolling screen-recording via OCR. See NOTES at the
bottom for spots that were ambiguous/cut off in the source video and need
your eyes on them.

In [ ]:
# Cell 1: fetch data + imports
# NOTE: the wget URL was cut off mid-scroll in the recording. You'll need to
# grab the full URL again (search your Inspirit AI course materials) or
# resupply the CSV another way.
!wget -O ./spotify_data_urls.csv 'https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions...'  # TODO: fix truncated URL
data_path = './spotify_data_urls.csv'

# General Imports:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import gdown
import ast
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
from IPython import display
import ipywidgets as widgets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Cell 2: more ML imports (this appeared as its own cell in the recording)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Cell 3: load data
data = pd.read_csv(data_path)

# NOTE: this feature list was cut off mid-scroll after 'acousticness'.
# It almost certainly continues with more audio features like
# 'instrumentalness', 'liveness', 'valence', 'tempo' etc. based on cell 5's
# reference to 'instrumentalness' -- check your original column list.
all_data = data[['Artist', 'Track', 'Year', 'url', 'Label', 'danceability',
                  'energy', 'speechiness', 'acousticness']]  # TODO: likely more columns
all_data.head()

In [ ]:
# Cell 4
data.head()

In [ ]:
# Cell 5: build combined text+numerical feature vectors
# NOTE: numerical_features list was cut off in the recording after
# 'acousticness' -- it references 'instrumentalness' so at minimum add that
# back in, and check for others (liveness, valence, tempo, etc.)
numerical_features = ['Label', 'Year', 'danceability', 'energy', 'speechiness',
                       'acousticness', 'instrumentalness']  # TODO: verify full list
text_features = ['Artist', 'Track']

def combine_features(row):
    """Loop through all of the features and make a string with all of them
    combined for one row."""
    combined_row = ''
    for feature in text_features:
        combined_row += str(row[feature])
        combined_row += ' '
    return combined_row[:-1]

# NOTE: 'fix_genres' is referenced but never defined in what was captured --
# this block only ran conditionally ("if 'genres' in text_features"), and
# since text_features above doesn't include 'genres', it likely didn't
# execute for you. Leaving it in case you added genres later.
if 'genres' in text_features:
    data['genres'] = data['genres'].fillna('')
    data['genres'] = data.apply(fix_genres, axis=1)  # TODO: fix_genres undefined

for feature in text_features:
    data[feature] = data[feature].fillna('')

data["combined_features"] = data.apply(combine_features, axis=1)

# putting into vectors
cv = CountVectorizer()
count_matrix = cv.fit_transform(data["combined_features"])  # creates a vector out of our combined features
text_vectors = count_matrix.toarray()

numerical = data[numerical_features].to_numpy()

# Scales numerical values so features like danceability and tempo don't
# outweigh everything else!
numerical = (numerical - numerical.min(axis=0)) / (numerical.max(axis=0) - numerical.min(axis=0))

song_vectors = np.concatenate((text_vectors, numerical), axis=1)
song_vectors

In [ ]:
# Cell 6
data.head(20)

In [ ]:
# Cell 7
data['combined_features']

In [ ]:
# Cell 8
song_vectors

After converting the data into vectors we can measure, there are a few
different ways to measure similarity:
- Manhattan Distance
- Euclidean Distance
- Cosine Distance

Calculate all similarity scores using Cosine Similarity.

In [ ]:
# Cell 9: similarity scoring function
def all_similarity(vectors, sim_metric='cosine'):
    if sim_metric == 'cosine':
        return cosine_similarity(vectors)
    else:
        # greater distance means the vectors are less similar, so we negate it
        return -pairwise_distances(vectors, metric=sim_metric)

# prints similarity matrix
sim_matrix = all_similarity(song_vectors, sim_metric='cosine')
print(sim_matrix)

In [ ]:
# Cell 10: visualize similarity matrix
fig, ax = plt.subplots()
im = ax.imshow(np.array(sim_matrix))

## Find Similarity between two inputted tracks
1. `find_title_from_index`: taking out the title from the given index
2. `find_artist_from_index`: taking out the artist from the index
3. `find_index_from_title`: finding index from title

In [ ]:
# Cell 11: lookup + scoring helpers
def find_title_from_index(index):
    return data["Track"][index]

def find_artist_from_index(index):
    return data["Artist"][index]

def find_index_from_title(track_name):
    return data.index[data.Track == track_name].values[0]

def similarity_score(track1, track2, vectors, metric='cosine'):
    """Calculates score of difference between 2 tracks."""
    if track1 == None or track2 == None:
        return None
    similarity_matrix = all_similarity(vectors, metric)
    track_1_index = find_index_from_title(track1)
    track_2_index = find_index_from_title(track2)
    score = similarity_matrix[track_1_index][track_2_index]
    return score

song = 'lithium'
song_index = find_index_from_title(song)

similarity_matrix = all_similarity(song_vectors)
# taking out the song's similarity scores from the matrix
song_similarity = similarity_matrix[song_index]

# lithium compared to every single song
print(song_similarity)

## 2 Approaches for getting the top five songs

### 1. Using a for loop
- Set K to 5 (top 5 songs)
- Loop, tracking the running max similarity score not already picked
  and not the song itself, and append the winning index each pass.

In [ ]:
# Cell 12: approach 1 -- manual loop
K = 5
song_indices = []

for i in range(0, K):
    current_max = 0
    current_index = None
    for j in range(len(song_similarity)):
        if song_similarity[j] > current_max and j not in song_indices and j != song_index:
            current_max = song_similarity[j]
            current_index = j
    song_indices.append(current_index)

print(song_indices)
# Expected output: [755, 2587, 1650, 2128, 3294]

### 2. Using Sorting
- Combine indices + scores with `enumerate`, sort descending by score,
  drop the first entry (the song matched against itself).

In [ ]:
# Cell 13
similar_songs = list(enumerate(song_similarity))
print(similar_songs)

In [ ]:
# Cell 14
sorted_similar_songs = sorted(similar_songs, key=lambda x: x[1], reverse=True)[1:]
print('sorted_similar_songs: ', sorted_similar_songs)
song_indices = [x[0] for x in sorted_similar_songs[:K]]
print(song_indices)

### Easier Way

In [ ]:
# Cell 15
sorted(song_similarity, reverse=True)[0:5]

In [ ]:
# Cell 16: drop the top match (itself)
final = sorted(song_similarity, reverse=True)[1:6]
print(final)

In [ ]:
# Cell 17: map scores back to indices
final_index = []
for i in final:
    final_index.append(np.where(similarity_matrix == i)[0][1])
print(final_index)

## Get Song title from Index

In [ ]:
# Cell 18
print('The top 5 recommended songs are: ')
for i, j in enumerate(final_index):
    print(i + 1, find_title_from_index(j))

## Make this into a reusable function
1. Have the user input a song
2. Find the song within the similarity matrix
3. Calculate the most similar songs (content-based filtering)
4. Recommend the top matches

In [ ]:
# Cell 19: form param version
Song = 'Party in the USA'  #@param {type:"string"}

In [ ]:
# Cell 20: dropdown widget version
options_song = data['Track']
Dropdown_ = widgets.Dropdown(
    options=options_song,
    description='UserSong:',
)

output = widgets.Output()
userinput = ''

def on_change(change):
    print(change['new'])
    userinput = change['new']
    return userinput

Dropdown_.observe(on_change, names='value')
display(Dropdown_)

## Use the user input to find the top 5 most similar songs
### 1. Get the user input

In [ ]:
# Cell 21
user_input_text = Song           # from the text param
user_input_drop = Dropdown_.value  # from the dropdown

print('Text: ', user_input_text)
print('Drop: ', user_input_drop)

### Find index from title & calculate song similarity

In [ ]:
# Cell 23 (labelled [23] in the recording -- cell 22 wasn't captured/likely blank)
song_index = find_index_from_title(user_input_drop)
song_similarity = similarity_matrix[song_index]

### Find top 5 most similar songs

In [ ]:
# Cell 24
sorted(song_similarity, reverse=True)[0:5]

final = sorted(song_similarity, reverse=True)[1:6]
print(final)

final_index = []
for i in final:
    final_index.append(np.where(similarity_matrix == i)[0][1])
print(final_index)

## Make this into a reusable function

In [ ]:
# Cell 25: SongRec function
# NOTE: this cell's body was partially obscured by an overlapping cell in
# the recording. Reconstructed based on the pattern of cells 23-24 combined
# -- double check the logic once you can run it.
def SongRec(userinput):
    song_index = find_index_from_title(userinput)
    similarity_matrix = all_similarity(song_vectors)
    song_similarity = similarity_matrix[song_index]

    final = sorted(song_similarity, reverse=True)[1:6]

    final_index = []
    for i in final:
        final_index.append(np.where(similarity_matrix == i)[0][1])

    final_songs = []
    for i, j in enumerate(final_index):
        final_songs.append(find_title_from_index(j))
    return final_songs

In [ ]:
# Cell 26/27: rebuild the dropdown and call SongRec on selection
options_song = data['Track']
Dropdown_ = widgets.Dropdown(
    options=options_song,
    description='UserSong:',
)

output = widgets.Output()
userinput = ''

def on_change(change):
    print(change['new'])
    userinput = change['new']
    return userinput

Dropdown_.observe(on_change, names='value')
display(Dropdown_)

SongRec(Dropdown_.value)
# Expected output for 'died in your arms':
# ['in these arms', 'something in your eyes', 'drowning in your eyes',
#  'into your arms', 'x']

## NOTES -- things to double check

1. **Cell 1 wget URL is truncated.** The recording scrolled past the end
   of the URL. You'll need the full data source URL from your course
   materials to re-download `spotify_data_urls.csv`.
2. **`numerical_features` list (Cell 5) is incomplete.** It's cut off
   right after `'acousticness'` in every frame that captured it, but the
   code clearly uses `'instrumentalness'` too. Common Spotify audio
   features you're likely missing: `liveness`, `valence`, `tempo`,
   `key`, `loudness`, `mode`. Check `data.columns` once you reload the
   CSV and compare against what's used elsewhere in the notebook.
3. **`fix_genres` function is referenced but never shown/defined** in any
   captured frame. It's only called conditionally (only if `'genres'` is
   in `text_features`, which it isn't in the captured code), so it may be
   dead code you can ignore -- or it may have existed in an earlier
   session/notebook you don't have. If you added genre features at some
   point, you'll need to rewrite this function.
4. **Cell numbering jumps from [21] to [23]** in the recording -- cell
   [22] was never visible in any frame, possibly just a rerun of [21] or
   a blank/markdown cell. If something errors around there, that's why.
5. **`SongRec` (Cell 25) body was partly obscured** by a UI overlay in the
   recording. I reconstructed it by pattern-matching cells 23-24, which
   do the same steps manually right before this function is defined --
   it should be logically equivalent, but run it and compare against the
   expected output at the bottom to confirm.